# 18 · Big Data: Hadoop, Spark, Spark SQL y MLlib

Este lab conecta conceptos de Big Data con práctica ejecutable en PySpark. Incluye HDFS/YARN/MapReduce como arquitectura, y Spark DataFrames, SQL y MLlib como práctica.

## Objetivos
- Entender por qué escalar verticalmente deja de ser suficiente.
- Distinguir storage distribuido (HDFS) y gestión de recursos (YARN).
- Entender MapReduce, Hive, Pig y su contexto histórico.
- Trabajar con Spark DataFrames y Spark SQL.
- Diseñar particiones y evitar shuffles innecesarios.
- Construir un Pipeline MLlib.
- Introducir Structured Streaming.

> En Google Colab trabajaremos en un Spark local de una máquina; la API es la misma, aunque un cluster real distribuye particiones entre executors.


## 1. Arquitectura Hadoop
**HDFS** divide archivos en bloques replicados entre nodos. El NameNode mantiene metadata; DataNodes almacenan bloques.

**YARN** separa gestión de recursos y ejecución: ResourceManager, NodeManagers y ApplicationMaster.

**MapReduce** expresa cómputo como `map → shuffle/sort → reduce`. Es robusto para batch, pero escribir pipelines iterativos de ML puede requerir múltiples lecturas/escrituras en disco.

**Hive** ofrece SQL sobre datos distribuidos; **Pig** introdujo un lenguaje de data flow. Spark ganó adopción por su DAG engine, procesamiento en memoria, APIs DataFrame/SQL/ML/streaming.


In [ ]:
!pip -q install pyspark
from pyspark.sql import SparkSession, functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StringIndexer, OneHotEncoder, StandardScaler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
spark=SparkSession.builder.master('local[*]').appName('AlexisTorresLabs-BigData').getOrCreate()
print(spark.version)

## 2. RDD vs DataFrame
RDD es una colección distribuida de objetos con transformaciones low-level. DataFrame agrega schema y permite al optimizador Catalyst reordenar/optimizar operaciones. Para la mayoría de ETL/analytics modernos, DataFrame/SQL es la opción principal.

Spark usa evaluación **lazy**: transformaciones como `select`/`filter` construyen un DAG y una acción (`count`, `collect`, write) dispara ejecución.


In [ ]:
import numpy as np, pandas as pd
rng=np.random.default_rng(42); n=100_000
pdf=pd.DataFrame({'id':np.arange(n),'region':rng.choice(['N','C','S'],n,p=[.2,.55,.25]),'edad':rng.integers(18,85,n),'ingreso':rng.lognormal(10.5,.7,n)})
p=1/(1+np.exp(-(-3+.04*(pdf.edad-40)+.000012*pdf.ingreso+(pdf.region=='C')*.5))); pdf['target']=rng.binomial(1,p)
sdf=spark.createDataFrame(pdf); sdf.printSchema(); sdf.show(5)

## 3. Transformaciones y plan de ejecución
`filter`, `select`, `withColumn`, `groupBy`, joins y window functions son equivalentes a muchas operaciones Pandas/SQL, pero deben pensarse en términos de movimiento distribuido.

Un **shuffle** redistribuye datos entre executors (groupBy, joins, distinct, orderBy) y suele ser costoso. Buen particionamiento, broadcast joins y evitar `collect()` sobre datasets grandes son reglas básicas.


In [ ]:
agg=(sdf.filter(F.col('edad')>=30).withColumn('log_ingreso',F.log1p('ingreso')).groupBy('region').agg(F.count('*').alias('n'),F.avg('ingreso').alias('avg_ingreso'),F.avg('target').alias('rate')).orderBy(F.desc('n')))
agg.show(); agg.explain(mode='formatted')

## 4. Spark SQL
DataFrame y SQL comparten el mismo motor. Esto facilita colaborar con equipos SQL y migrar lógica analítica.


In [ ]:
sdf.createOrReplaceTempView('personas')
spark.sql('''SELECT region, COUNT(*) n, percentile_approx(ingreso,0.5) mediana_ingreso, AVG(target) tasa FROM personas GROUP BY region ORDER BY n DESC''').show()

## 5. Particiones y formatos
Parquet/Delta son columnares: permiten predicate pushdown y leer solo columnas necesarias. Particionar por una columna de baja/moderada cardinalidad (p.ej. fecha/año) puede reducir scan; particionar por ID único crea millones de archivos pequeños.

Problemas típicos:
- **small files**;
- data skew (una key concentra filas);
- demasiadas particiones o muy pocas;
- joins grandes que explotan memoria;
- Python UDF cuando existen funciones Spark nativas.


In [ ]:
print('particiones actuales',sdf.rdd.getNumPartitions()); repart=sdf.repartition(6,'region'); print('repartition',repart.rdd.getNumPartitions())
# En un cluster real:
# repart.write.mode('overwrite').partitionBy('region').parquet('/ruta/dataset')

## 6. Machine Learning con MLlib
MLlib usa una columna vector de features. `Pipeline` encadena indexación, encoding, assembler y estimador, igual que la idea de sklearn pero distribuida.


In [ ]:
train,test=sdf.randomSplit([.8,.2],seed=42)
idx=StringIndexer(inputCol='region',outputCol='region_idx',handleInvalid='keep'); ohe=OneHotEncoder(inputCols=['region_idx'],outputCols=['region_ohe'])
vec=VectorAssembler(inputCols=['edad','ingreso','region_ohe'],outputCol='features'); clf=LogisticRegression(featuresCol='features',labelCol='target',maxIter=40,regParam=.01)
pipe=Pipeline(stages=[idx,ohe,vec,clf]); model=pipe.fit(train); pred=model.transform(test)
evalr=BinaryClassificationEvaluator(labelCol='target',rawPredictionCol='rawPrediction',metricName='areaUnderROC'); print('AUC',evalr.evaluate(pred)); pred.select('target','probability','prediction').show(8,truncate=False)

## 7. Tuning distribuido
`CrossValidator` entrena varias configuraciones/folds; en grandes datasets puede multiplicar brutalmente el costo. Random search o frameworks como Optuna/Hyperopt + MLflow suelen ser más eficientes.


In [ ]:
grid=ParamGridBuilder().addGrid(clf.regParam,[.0,.01,.1]).addGrid(clf.elasticNetParam,[0.,.5,1.]).build()
cv=CrossValidator(estimator=pipe,estimatorParamMaps=grid,evaluator=evalr,numFolds=3,parallelism=2,seed=42)
# Descomenta para ejecutar (varios entrenamientos):
# cvmodel=cv.fit(train); print(evalr.evaluate(cvmodel.transform(test)))

## 8. Structured Streaming
Spark Streaming modela un stream como una tabla que crece. Fuentes comunes: Kafka, files, sockets; sinks: Delta, Kafka, consola. Watermarks ayudan con eventos tardíos y estado.

```python
stream = spark.readStream.format('kafka')...load()
result = stream.withWatermark('event_time','10 minutes').groupBy(window('event_time','5 minutes')).count()
result.writeStream.format('delta')...start()
```

## Cuándo usar Spark y cuándo no
- Pandas/Polars si los datos caben cómodamente en una máquina.
- DuckDB para analytics local sorprendentemente grande.
- Dask/Ray para Python distribuido flexible.
- Spark cuando ya existe lakehouse/cluster, datasets masivos, SQL distribuido, ETL multiusuario o MLlib a escala.

## Ejercicios
1. Compara `groupBy` con y sin repartition.
2. Crea un join con tabla pequeña usando `broadcast`.
3. Guarda/lee Parquet particionado.
4. Usa Spark SQL window functions.
5. Cambia Logistic por RandomForestClassifier.
6. Implementa un pipeline de clasificación multiclase.
7. Simula streaming con `rate` source.
8. Investiga Delta Lake, Databricks, MLflow y feature tables.
